# 06 — Validação com LLM: personas, tensões, JTBD e metadados

Este notebook usa a API da Groq como camada de **validação qualitativa** dos notebooks 04 e 05.

Objetivo: comparar o que o modelo estatístico/dicionários encontraram com uma segunda leitura de LLM, para verificar se as personas, tensões, motivadores, barreiras e Jobs-to-be-Done fazem sentido.

O notebook gera:

- `llm_persona_validation.csv`: validação das personas.
- `llm_doc_classification.csv`: classificação independente de uma amostra de entrevistas.
- `llm_rule_based_comparison.csv`: comparação LLM vs notebooks 04/05.
- `llm_metadata_inference.csv`: tentativa de reduzir `nao_inferido` com evidência.
- `llm_dimension_dictionary_review.csv`: revisão dos dicionários/dimensões.
- `relatorio_validacao_llm.md`: relatório final.

> A LLM não substitui revisão humana; ela ajuda a auditar e enriquecer a análise.

## 0. Setup

Crie um `.env` na raiz do projeto:

```text
GROQ_API_KEY=sua_chave
GROQ_MODEL=llama-3.1-8b-instant
```

Se o modelo não estiver disponível, troque `GROQ_MODEL`.

In [ ]:
# !pip install -q groq python-dotenv pandas numpy tqdm pyarrow plotly

In [ ]:
from pathlib import Path
import os, re, json, time, hashlib
from getpass import getpass
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 240)

## 1. Localizar projeto e carregar bases

In [ ]:
def find_project_root():
    here = Path.cwd().resolve()
    candidates = [here, here.parent, here.parent.parent, Path('/mnt/data'), Path('/content'), Path('/content/kyrav2')]
    for c in candidates:
        if not c.exists():
            continue
        if (c/'data'/'processed').exists() and (c/'notebooks').exists():
            return c
        if (c/'kyrav2'/'data'/'processed').exists():
            return c/'kyrav2'
    return here

ROOT = find_project_root()
OUT_DIR = ROOT/'outputs'/'llm_validacao'
TABLE_DIR = OUT_DIR/'tables'
REPORT_DIR = OUT_DIR/'reports'
CACHE_DIR = OUT_DIR/'cache_groq'
PROMPT_DIR = OUT_DIR/'prompts'
for d in [OUT_DIR, TABLE_DIR, REPORT_DIR, CACHE_DIR, PROMPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT)
print('OUT_DIR:', OUT_DIR)

In [ ]:
def read_first_csv(paths, required=False):
    for p in paths:
        p = Path(p)
        if p.exists():
            print('Carregando:', p)
            return pd.read_csv(p)
    if required:
        raise FileNotFoundError('Não encontrei nenhum arquivo em: ' + ' | '.join(map(str, paths)))
    return pd.DataFrame()

def read_first_table(paths, required=False):
    for p in paths:
        p = Path(p)
        if p.exists():
            print('Carregando:', p)
            return pd.read_parquet(p) if p.suffix.lower()=='.parquet' else pd.read_csv(p)
    if required:
        raise FileNotFoundError('Não encontrei nenhum arquivo em: ' + ' | '.join(map(str, paths)))
    return pd.DataFrame()

P = ROOT/'outputs'/'personas'/'tables'
P2 = ROOT/'outputs'/'personas'/'solutions'/'macro_k3'/'tables'
T = ROOT/'outputs'/'tensoes_jtbd'/'tables'
LOCAL = Path('/mnt/data')

personas_doc = read_first_csv([P/'personas_doc_level.csv', P2/'personas_doc_level.csv', LOCAL/'personas_doc_level.csv'])
personas_summary = read_first_csv([P/'personas_summary.csv', P2/'personas_summary.csv', LOCAL/'personas_summary.csv'])
personas_lifts = read_first_csv([P/'personas_dimension_lifts.csv', P2/'personas_dimension_lifts.csv', LOCAL/'personas_dimension_lifts.csv'])
personas_quotes = read_first_csv([P/'personas_evidence_quotes.csv', P2/'personas_evidence_quotes.csv', LOCAL/'personas_evidence_quotes.csv'])
metadata_diag = read_first_csv([P/'persona_metadata_diagnostics.csv', P2/'persona_metadata_diagnostics.csv', LOCAL/'persona_metadata_diagnostics.csv'])
tensoes_scores = read_first_csv([T/'entrevistas_scores_tensoes_jtbd.csv', LOCAL/'entrevistas_scores_tensoes_jtbd.csv'])
jtbd_resumo = read_first_csv([T/'jtbd_resumo.csv', LOCAL/'jtbd_resumo.csv'])
mot_bar_resumo = read_first_csv([T/'motivadores_barreiras_resumo.csv', LOCAL/'motivadores_barreiras_resumo.csv'])
evid_dim = read_first_csv([T/'evidencias_citaveis_por_dimensao.csv', LOCAL/'evidencias_citaveis_por_dimensao.csv'])
interviews_docs = read_first_table([ROOT/'data'/'processed'/'interviews_docs_preprocessed.parquet', ROOT/'data'/'processed'/'interviews_docs_preprocessed.csv', LOCAL/'interviews_docs_preprocessed.parquet', LOCAL/'interviews_docs_preprocessed.csv'])

for name, df in {'personas_doc': personas_doc, 'personas_summary': personas_summary, 'personas_lifts': personas_lifts, 'personas_quotes': personas_quotes, 'metadata_diag': metadata_diag, 'tensoes_scores': tensoes_scores, 'jtbd_resumo': jtbd_resumo, 'mot_bar_resumo': mot_bar_resumo, 'evid_dim': evid_dim, 'interviews_docs': interviews_docs}.items():
    print(f'{name:22s}', df.shape)

## 2. Nomes manuais das personas e base unificada

In [ ]:
PERSONA_NAME_MAP = {
    'persona_00': 'Cuidadora Consciente',
    'macro_k3_p00': 'Cuidadora Consciente',
    'persona_01': 'Comparadora Omnicanal',
    'macro_k3_p01': 'Comparadora Omnicanal',
    'persona_02': 'Compradora Social e Presenteadora',
    'macro_k3_p02': 'Compradora Social e Presenteadora',
}

PERSONA_DEFINITIONS = {
    'Cuidadora Consciente': {'descricao': 'Cuidado pessoal, pele/corpo, naturalidade, eficácia, inovação e representatividade. Precisa de prova e segurança.', 'sinais': ['pele','corpo','cuidado','natural','sustentável','ingredientes','resultado','eficácia','inovação','representatividade','dúvida sobre claims']},
    'Comparadora Omnicanal': {'descricao': 'Compara marcas e canais; avalia loja, consultora, WhatsApp, preço, disponibilidade, confiança e custo-benefício.', 'sinais': ['Natura','Boticário','Avon','loja','consultora','preço','promoção','frete','sacola','disponibilidade','online']},
    'Compradora Social e Presenteadora': {'descricao': 'Compra em contexto social/familiar: presente, indicação, cuidado com outros, WhatsApp, consultora, família/amigas.', 'sinais': ['presente','mãe','filho','família','amiga','indicação','WhatsApp','consultora','ocasião','aniversário','Dia das Mães']},
}

def apply_manual_names(df):
    df = df.copy()
    if df.empty:
        return df
    df['persona_nome_manual'] = np.nan
    if 'persona_id' in df.columns:
        df['persona_nome_manual'] = df['persona_id'].map(PERSONA_NAME_MAP)
    if 'persona_nome_auto' in df.columns:
        auto = df['persona_nome_auto'].astype(str).str.lower()
        df.loc[df['persona_nome_manual'].isna() & auto.str.contains('consciente|natural|pele|corpo', regex=True), 'persona_nome_manual'] = 'Cuidadora Consciente'
        df.loc[df['persona_nome_manual'].isna() & auto.str.contains('comparadora|marcas|canal|loja|consultora', regex=True), 'persona_nome_manual'] = 'Comparadora Omnicanal'
        df.loc[df['persona_nome_manual'].isna() & auto.str.contains('social|familiar|indicação|presenteadora|presente', regex=True), 'persona_nome_manual'] = 'Compradora Social e Presenteadora'
        df['persona_nome_manual'] = df['persona_nome_manual'].fillna(df['persona_nome_auto'])
    return df

for nm in ['personas_doc','personas_summary','personas_lifts','personas_quotes','metadata_diag','tensoes_scores']:
    if nm in globals():
        globals()[nm] = apply_manual_names(globals()[nm])

if not interviews_docs.empty and not personas_doc.empty and 'doc_id' in interviews_docs.columns:
    text_col = 'texto_clean' if 'texto_clean' in interviews_docs.columns else ('texto_raw' if 'texto_raw' in interviews_docs.columns else None)
    keep = ['doc_id']
    for c in [text_col, 'projeto','pais','marca_foco','publico','tipo_sessao','idioma_original','resumo_executivo_curto','evidencia_citavel_doc']:
        if c and c in interviews_docs.columns and c not in keep:
            keep.append(c)
    doc_text = interviews_docs[keep].copy()
    if text_col:
        doc_text = doc_text.rename(columns={text_col: 'texto_entrevista'})
    base_docs = personas_doc.merge(doc_text, on='doc_id', how='left', suffixes=('', '_raw'))
else:
    base_docs = personas_doc.copy()
    base_docs['texto_entrevista'] = ''

if not tensoes_scores.empty and 'doc_id' in tensoes_scores.columns:
    cols = [c for c in ['doc_id','motivadores_total_per_1k','barreiras_total_per_1k','saldo_compra_rejeicao','intensidade_tensao','motivador_principal','barreira_principal','jtbd_principal'] if c in tensoes_scores.columns]
    base_docs = base_docs.merge(tensoes_scores[cols], on='doc_id', how='left')

print(base_docs.shape)
display(base_docs.head(3))

## 3. Configurar Groq e helpers JSON

In [ ]:
try:
    from groq import Groq
except ImportError:
    raise ImportError('Instale com: pip install groq')

api_key = os.getenv('GROQ_API_KEY')
if not api_key:
    api_key = getpass('Cole sua GROQ_API_KEY aqui. Ela não será salva no notebook: ')
    if api_key:
        os.environ['GROQ_API_KEY'] = api_key

GROQ_MODEL = os.getenv('GROQ_MODEL', 'llama-3.1-8b-instant')
RUN_LLM = bool(os.getenv('GROQ_API_KEY'))
client = Groq(api_key=os.getenv('GROQ_API_KEY')) if RUN_LLM else None
print('RUN_LLM:', RUN_LLM, '| GROQ_MODEL:', GROQ_MODEL)

In [ ]:
def stable_hash(text):
    return hashlib.md5(text.encode('utf-8')).hexdigest()

def extract_json(text):
    if text is None:
        return None
    text = text.strip()
    clean = re.sub(r'^```(?:json)?|```$', '', text, flags=re.I | re.M).strip()
    for candidate in [text, clean]:
        try:
            return json.loads(candidate)
        except Exception:
            pass
    for a, b in [('{','}'), ('[',']')]:
        i, j = text.find(a), text.rfind(b)
        if i >= 0 and j > i:
            try:
                return json.loads(text[i:j+1])
            except Exception:
                pass
    return {'parse_error': True, 'raw': text[:5000]}

def truncate_text(text, max_chars=7000):
    text = '' if pd.isna(text) else str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    return text[:half] + '\n[...trecho central omitido...]\n' + text[-half:]

def groq_json_call(task_name, prompt, system='Você é uma pesquisadora sênior de insights qualitativos. Responda apenas em JSON válido.', temperature=0.0, max_tokens=1800, sleep_s=0.7, force_refresh=False):
    payload = json.dumps({'system': system, 'prompt': prompt, 'model': GROQ_MODEL, 'temperature': temperature}, ensure_ascii=False)
    key = f'{task_name}_{stable_hash(payload)}'
    cache_path = CACHE_DIR / f'{key}.json'
    prompt_path = PROMPT_DIR / f'{key}.txt'
    prompt_path.write_text(prompt, encoding='utf-8')
    if cache_path.exists() and not force_refresh:
        return json.loads(cache_path.read_text(encoding='utf-8'))
    if not RUN_LLM:
        out = {'task_name': task_name, 'skipped': True, 'reason': 'GROQ_API_KEY ausente. Prompt salvo em outputs/llm_validacao/prompts.', 'prompt_file': str(prompt_path)}
        cache_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')
        return out
    resp = client.chat.completions.create(model=GROQ_MODEL, temperature=temperature, max_tokens=max_tokens, messages=[{'role':'system','content':system},{'role':'user','content':prompt}])
    raw = resp.choices[0].message.content
    out = {'task_name': task_name, 'model': GROQ_MODEL, 'raw': raw, 'parsed': extract_json(raw)}
    cache_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')
    time.sleep(sleep_s)
    return out

## 4. Validar personas com LLM

In [ ]:
def get_persona_quotes(persona_name, n=7):
    if personas_quotes.empty:
        return []
    q = personas_quotes.copy()
    if 'persona_nome_manual' in q.columns:
        q = q[q['persona_nome_manual'] == persona_name]
    col = 'excerpt' if 'excerpt' in q.columns else ('evidencia' if 'evidencia' in q.columns else None)
    return [truncate_text(x, 1200) for x in q[col].dropna().head(n).tolist()] if col else []

def get_persona_lifts(persona_name, n=8):
    if personas_lifts.empty:
        return []
    df = personas_lifts.copy()
    if 'persona_nome_manual' in df.columns:
        df = df[df['persona_nome_manual'] == persona_name]
    sort_cols = [c for c in ['lift_vs_global','lift','score','media_per_1k'] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols[0], ascending=False)
    keep = [c for c in ['dimensao','dimension','lift_vs_global','lift','media_persona','media_global','score'] if c in df.columns]
    return df[keep].head(n).to_dict('records') if keep else []

def get_metadata_diag(persona_name):
    if metadata_diag.empty:
        return {}
    df = metadata_diag.copy()
    if 'persona_nome_manual' in df.columns:
        df = df[df['persona_nome_manual'] == persona_name]
    return df.head(1).to_dict('records')[0] if len(df) else {}

persona_validation_results = []
for persona_name, definition in tqdm(PERSONA_DEFINITIONS.items(), desc='Validando personas'):
    content = {
        'persona_manual': persona_name,
        'descricao_esperada': definition,
        'dimensoes_que_diferenciam': get_persona_lifts(persona_name),
        'diagnostico_metadados': get_metadata_diag(persona_name),
        'trechos_representativos': get_persona_quotes(persona_name),
    }
    instr = 'Valide qualitativamente a persona abaixo. Diga se o nome faz sentido, se há risco de artefato de projeto/amostra e como usar em negócio. Responda apenas em JSON com chaves: persona, nome_faz_sentido, coerencia_qualitativa_1a5, risco_artefato_projeto_1a5, resumo_julgamento, evidencias_que_sustentam, evidencias_ou_sinais_fracos, ajuste_nome_sugerido, ajustes_metodologicos, como_usar_em_negocio.'
    prompt = instr + '\n\n' + json.dumps(content, ensure_ascii=False, indent=2)
    out = groq_json_call('persona_validation', prompt, max_tokens=2200)
    parsed = out.get('parsed', out)
    persona_validation_results.append(parsed if isinstance(parsed, dict) else {'persona': persona_name, 'raw': parsed})

llm_persona_validation = pd.DataFrame(persona_validation_results)
llm_persona_validation.to_csv(TABLE_DIR/'llm_persona_validation.csv', index=False)
display(llm_persona_validation)

## 5. Classificar uma amostra de entrevistas com LLM e comparar com notebooks 04/05

In [ ]:
MAX_DOCS_PER_PERSONA = 4
MAX_CHARS_PER_INTERVIEW = 6500

sample_docs = base_docs.copy()
if 'persona_nome_manual' in sample_docs.columns:
    sample_docs = sample_docs.dropna(subset=['persona_nome_manual']).groupby('persona_nome_manual', group_keys=False).apply(lambda x: x.sample(min(MAX_DOCS_PER_PERSONA, len(x)), random_state=42)).reset_index(drop=True)
else:
    sample_docs = sample_docs.sample(min(12, len(sample_docs)), random_state=42)

MOTIVADOR_OPTIONS = ['preco_valor_promocao','eficacia_resultado_prova','sensorial_experiencia','marca_confianca_memoria','natural_sustentavel_ingredientes','presente_status_estetica','canal_conveniencia_consultora','indicacao_social_familiar','nao_identificado']
BARREIRA_OPTIONS = ['preco_alto_sem_valor','duvida_eficacia_claim','sensorial_negativo','canal_friccao_compra','marca_comparacao_desvantagem','sustentabilidade_ceticismo','indecisao_excesso_opcoes','nao_identificado']
JTBD_OPTIONS = ['presentear_e_demonstrar_cuidado','autoestima_e_expressao_identidade','praticidade_rotina_rapida','eficacia_tratamento_resolver_problema','cuidado_familiar_e_social','economizar_sem_perder_qualidade','experiencia_sensorial_prazer_ritual','pertencimento_recomendacao_comunidade','compra_conveniente_e_acesso','nao_identificado']

display(sample_docs[[c for c in ['doc_id','persona_nome_manual','projeto','pais','marca_foco','publico','motivador_principal','barreira_principal','jtbd_principal'] if c in sample_docs.columns]])

In [ ]:
llm_doc_results = []
for _, row in tqdm(sample_docs.iterrows(), total=len(sample_docs), desc='Classificando entrevistas'):
    text = truncate_text(row.get('texto_entrevista', ''), MAX_CHARS_PER_INTERVIEW) or truncate_text(row.get('evidencia_citavel_doc', ''), MAX_CHARS_PER_INTERVIEW)
    content = {'doc_id': row.get('doc_id'), 'personas': PERSONA_DEFINITIONS, 'motivadores': MOTIVADOR_OPTIONS, 'barreiras': BARREIRA_OPTIONS, 'jtbd': JTBD_OPTIONS, 'entrevista': text}
    instr = 'Classifique a entrevista de forma independente. Responda apenas em JSON com chaves: doc_id, persona_llm_principal, persona_aderencia_pct, motivador_principal_llm, barreira_principal_llm, jtbd_principal_llm, tensao_compra, confianca_1a5, evidencias, comentario.'
    prompt = instr + '\n\n' + json.dumps(content, ensure_ascii=False, indent=2)
    out = groq_json_call('doc_independent_classification', prompt, max_tokens=2400)
    parsed = out.get('parsed', out)
    llm_doc_results.append(parsed if isinstance(parsed, dict) else {'doc_id': row.get('doc_id'), 'raw': parsed})

llm_doc_classification = pd.DataFrame(llm_doc_results)
llm_doc_classification.to_csv(TABLE_DIR/'llm_doc_classification.csv', index=False)
display(llm_doc_classification.head())

In [ ]:
if not llm_doc_classification.empty:
    comp_cols = [c for c in ['doc_id','persona_nome_manual','motivador_principal','barreira_principal','jtbd_principal','motivadores_total_per_1k','barreiras_total_per_1k'] if c in base_docs.columns]
    comparison = llm_doc_classification.merge(base_docs[comp_cols], on='doc_id', how='left')
    def same_norm(a,b):
        if pd.isna(a) or pd.isna(b): return np.nan
        return str(a).strip().lower() == str(b).strip().lower()
    pairs = [('persona_llm_principal','persona_nome_manual','match_persona'), ('motivador_principal_llm','motivador_principal','match_motivador'), ('barreira_principal_llm','barreira_principal','match_barreira'), ('jtbd_principal_llm','jtbd_principal','match_jtbd')]
    for a,b,m in pairs:
        if a in comparison.columns and b in comparison.columns:
            comparison[m] = comparison.apply(lambda r: same_norm(r[a], r[b]), axis=1)
    comparison.to_csv(TABLE_DIR/'llm_rule_based_comparison.csv', index=False)
    metric_cols = [c for c in ['match_persona','match_motivador','match_barreira','match_jtbd'] if c in comparison.columns]
    if metric_cols:
        agreement_summary = comparison[metric_cols].mean(numeric_only=True).reset_index()
        agreement_summary.columns = ['metrica','taxa_match']
        agreement_summary.to_csv(TABLE_DIR/'llm_agreement_summary.csv', index=False)
        display(agreement_summary)
    display(comparison[[c for c in ['doc_id','persona_nome_manual','persona_llm_principal','match_persona','motivador_principal','motivador_principal_llm','match_motivador','barreira_principal','barreira_principal_llm','match_barreira','jtbd_principal','jtbd_principal_llm','match_jtbd','comentario'] if c in comparison.columns]])
else:
    comparison = pd.DataFrame()
    print('Sem resultados de LLM para comparar.')

## 6. Inferir metadados ausentes (`nao_inferido`)

In [ ]:
MAX_META_DOCS = 15
meta_candidates = base_docs.copy()
meta_cols = [c for c in ['pais','marca_foco','publico','tipo_sessao'] if c in meta_candidates.columns]
if meta_cols:
    mask = pd.Series(False, index=meta_candidates.index)
    for c in meta_cols:
        mask = mask | meta_candidates[c].astype(str).str.contains('nao_inferido|nan|None', case=False, na=False)
    meta_candidates = meta_candidates[mask].sample(min(MAX_META_DOCS, int(mask.sum())), random_state=42) if mask.sum() else meta_candidates.head(0)
else:
    meta_candidates = meta_candidates.head(0)

metadata_results = []
for _, row in tqdm(meta_candidates.iterrows(), total=len(meta_candidates), desc='Inferindo metadados'):
    content = {'doc_id': row.get('doc_id'), 'metadados_atuais': {c: row.get(c) for c in ['projeto','pais','marca_foco','publico','tipo_sessao']}, 'texto': truncate_text(row.get('texto_entrevista', ''), 5500)}
    instr = 'Inferir metadados com base apenas no texto. Se não houver evidência, use nao_inferido. Responda apenas em JSON com chaves: doc_id, pais_inferido, marca_foco_inferida, publico_inferido, tipo_sessao_inferido, confianca_pais_1a5, confianca_marca_1a5, confianca_publico_1a5, confianca_tipo_1a5, evidencias, observacao.'
    prompt = instr + '\n\n' + json.dumps(content, ensure_ascii=False, indent=2)
    out = groq_json_call('metadata_inference', prompt, max_tokens=2000)
    parsed = out.get('parsed', out)
    metadata_results.append(parsed if isinstance(parsed, dict) else {'doc_id': row.get('doc_id'), 'raw': parsed})

llm_metadata_inference = pd.DataFrame(metadata_results)
llm_metadata_inference.to_csv(TABLE_DIR/'llm_metadata_inference.csv', index=False)
display(llm_metadata_inference.head(20))

## 7. Revisar dicionários/dimensões

In [ ]:
MAX_DIMENSIONS = 12
EXAMPLES_PER_DIM = 4
sample = []
if not evid_dim.empty and 'dimensao' in evid_dim.columns:
    for dim, g in evid_dim.groupby('dimensao'):
        g2 = g.dropna(subset=['evidencia']) if 'evidencia' in g.columns else g
        if len(g2):
            ex = g2.head(EXAMPLES_PER_DIM)
            sample.append({'dimensao': dim, 'exemplos': ex[[c for c in ['doc_id','evidencia','hits_na_frase','score_doc_per_1k','persona_nome_auto'] if c in ex.columns]].to_dict('records')})
    sample = sample[:MAX_DIMENSIONS]

instr = 'Revise dimensões criadas por dicionários em entrevistas. Para cada dimensão, diga se os exemplos sustentam a categoria, se há falso positivo, termos bons, termos a remover e termos novos. Responda apenas em JSON como lista.'
prompt = instr + '\n\n' + json.dumps(sample, ensure_ascii=False, indent=2)[:18000]
out = groq_json_call('dimension_dictionary_review', prompt, max_tokens=2600)
parsed = out.get('parsed', out)
llm_dimension_review = pd.DataFrame(parsed if isinstance(parsed, list) else [parsed])
llm_dimension_review.to_csv(TABLE_DIR/'llm_dimension_dictionary_review.csv', index=False)
display(llm_dimension_review)

## 8. Revisão executiva dos achados

In [ ]:
def df_to_records(df, max_rows=20):
    if df is None or df.empty:
        return []
    return df.head(max_rows).replace({np.nan: None}).to_dict('records')

context = {
    'personas_summary': df_to_records(personas_summary, 10),
    'metadata_diagnostics': df_to_records(metadata_diag, 10),
    'jtbd_resumo': df_to_records(jtbd_resumo, 15),
    'motivadores_barreiras_resumo': df_to_records(mot_bar_resumo, 20),
    'persona_validation_llm': df_to_records(llm_persona_validation, 10),
    'agreement_summary': df_to_records(agreement_summary, 10) if 'agreement_summary' in globals() else [],
    'dimension_review': df_to_records(llm_dimension_review, 12) if 'llm_dimension_review' in globals() else [],
}

instr = 'Revise criticamente os resultados abaixo. Gere uma avaliação executiva honesta para negócio. Responda apenas em JSON com chaves: veredito_geral, insights_mais_fortes, insights_que_exigem_cautela, principais_entraves_metodologicos, aplicacoes_llm_mais_valiosas, melhorias_prioritarias_para_proxima_versao, como_apresentar_para_negocio, frase_de_conclusao.'
prompt = instr + '\n\n' + json.dumps(context, ensure_ascii=False, indent=2)[:22000]
out = groq_json_call('executive_findings_review', prompt, max_tokens=2600)
parsed = out.get('parsed', out)
llm_findings_review = pd.DataFrame([parsed] if isinstance(parsed, dict) else [{'raw': parsed}])
llm_findings_review.to_csv(TABLE_DIR/'llm_findings_review.csv', index=False)
display(llm_findings_review.T)

## 9. Classificador LLM para nova entrevista

In [ ]:
NEW_INTERVIEW_TEXT = 'Cole aqui uma nova entrevista ou trecho longo de entrevista.'

if NEW_INTERVIEW_TEXT and 'Cole aqui' not in NEW_INTERVIEW_TEXT:
    content = {'personas': PERSONA_DEFINITIONS, 'motivadores': MOTIVADOR_OPTIONS, 'barreiras': BARREIRA_OPTIONS, 'jtbd': JTBD_OPTIONS, 'entrevista': truncate_text(NEW_INTERVIEW_TEXT, 9000)}
    instr = 'Classifique a nova entrevista. Responda apenas em JSON com persona_principal, persona_aderencia_pct, motivadores, barreiras, jtbd_principal, jtbd_secundarios, tensao_compra, oportunidades, alertas_de_interpretacao.'
    prompt = instr + '\n\n' + json.dumps(content, ensure_ascii=False, indent=2)
    nova = groq_json_call('new_interview_classifier', prompt, max_tokens=2400)
    display(nova.get('parsed', nova))
else:
    print('Cole uma entrevista em NEW_INTERVIEW_TEXT para rodar a classificação LLM.')

## 10. Relatório Markdown final

In [ ]:
def as_bullets(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return '- n/d'
    if isinstance(x, list):
        return '\n'.join(f'- {i}' for i in x)
    return f'- {x}'

lines = []
lines.append('# Relatório — Validação LLM das personas, tensões e JTBD\n')
lines.append(f'Modelo usado: `{GROQ_MODEL}`\n')
lines.append('## Objetivo\nValidar qualitativamente personas, motivadores, barreiras e JTBD dos notebooks 04/05.\n')
lines.append('## Validação das personas\n')
for _, r in llm_persona_validation.iterrows():
    lines.append(f"### {r.get('persona', 'persona')}\n")
    lines.append(f"- Coerência qualitativa: {r.get('coerencia_qualitativa_1a5', 'n/d')}/5")
    lines.append(f"- Risco de artefato de projeto: {r.get('risco_artefato_projeto_1a5', 'n/d')}/5")
    lines.append(f"- Julgamento: {r.get('resumo_julgamento', 'n/d')}\n")

if 'agreement_summary' in globals() and not agreement_summary.empty:
    lines.append('## Comparação LLM vs notebooks 04/05\n')
    for _, r in agreement_summary.iterrows():
        lines.append(f"- {r['metrica']}: {r['taxa_match']:.1%}")
    lines.append('')

if 'llm_findings_review' in globals() and not llm_findings_review.empty:
    lines.append('## Revisão executiva\n')
    row = llm_findings_review.iloc[0].to_dict()
    for k, v in row.items():
        lines.append(f"### {str(k).replace('_',' ').capitalize()}\n{as_bullets(v)}\n")

lines.append('## Arquivos gerados\n')
for f in sorted(TABLE_DIR.glob('*.csv')):
    lines.append(f'- `{f}`')

report_path = REPORT_DIR/'relatorio_validacao_llm.md'
report_path.write_text('\n'.join(lines), encoding='utf-8')
print('Relatório salvo em:', report_path)
print('\n'.join(lines[:80]))

## 11. Como usar o notebook 06

Use como auditoria qualitativa:

- Se a LLM concorda com as personas, isso fortalece a leitura.
- Se a LLM discorda, revise manualmente os trechos.
- Se ela apontar risco de artefato, rode a análise por projeto ou melhore a tabela de metadados.
- Se ela encontrar falsos positivos, ajuste os dicionários dos notebooks 04/05.
- O uso mais interessante não é sentimento positivo/negativo/neutro, mas sim validação qualitativa, extração de evidências, inferência de metadados, classificação de novas entrevistas e geração de recomendações.